Для чего этот ноутбук?
- проверить, что curated-слой успешно сформирован после ETL;
- убедиться в корректности типов данных, диапазонов значений и отсутствии критичных NULL;
- подтвердить готовность данных к аналитике в Superset и к ML-задачам.

In [1]:
import os
import pandas as pd
import s3fs
from dotenv import load_dotenv

load_dotenv()

S3_BUCKET_CURATED = os.getenv("S3_BUCKET_CURATED", "curated")
MINIO_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID", "minioadmin")
MINIO_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin123")
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "minio:9000")
MINIO_SECURE = os.getenv("MINIO_SECURE", "false").lower() == "true"


def get_s3_fs():
    protocol = "https" if MINIO_SECURE else "http"
    return s3fs.S3FileSystem(
        key=MINIO_ACCESS_KEY,
        secret=MINIO_SECRET_KEY,
        client_kwargs={"endpoint_url": f"{protocol}://{MINIO_ENDPOINT}"},
    )


fs = get_s3_fs()

In [2]:
production_clean = pd.read_parquet(
    f"s3://{S3_BUCKET_CURATED}/clean/production/production_clean.parquet",
    filesystem=fs,
)

well_telemetry_clean = pd.read_parquet(
    f"s3://{S3_BUCKET_CURATED}/clean/well_telemetry/well_telemetry_clean.parquet",
    filesystem=fs,
)

mart_production_daily = pd.read_parquet(
    f"s3://{S3_BUCKET_CURATED}/marts/mart_production_daily.parquet",
    filesystem=fs,
)

mart_well_kpi = pd.read_parquet(
    f"s3://{S3_BUCKET_CURATED}/marts/mart_well_kpi.parquet",
    filesystem=fs,
)

In [3]:
tables_info = pd.DataFrame({
    "table_name": [
        "production_clean",
        "well_telemetry_clean",
        "mart_production_daily",
        "mart_well_kpi"
    ],
    "rows": [
        len(production_clean),
        len(well_telemetry_clean),
        len(mart_production_daily),
        len(mart_well_kpi)
    ],
    "columns": [
        production_clean.shape[1],
        well_telemetry_clean.shape[1],
        mart_production_daily.shape[1],
        mart_well_kpi.shape[1]
    ]
})

tables_info

,table_name,rows,columns
0,production_clean,150,10
1,well_telemetry_clean,48,11
2,mart_production_daily,30,10
3,mart_well_kpi,5,13


In [4]:
production_clean.dtypes

production_id              int64
well_id                    int64
date              datetime64[ns]
oil_ton                  float64
gas_m3                   float64
water_m3                 float64
energy_kwh               float64
downtime_hours           float64
temperature              float64
pressure                 float64
dtype: object

In [5]:
well_telemetry_clean.dtypes

record_id                  int64
well_id                    int64
timestamp         datetime64[ns]
pump_speed_rpm           float64
pump_current             float64
pressure_in              float64
pressure_out             float64
temperature              float64
vibration                float64
oil_flow_rate            float64
date                      object
dtype: object

In [6]:
mart_production_daily.dtypes

date                    datetime64[ns]
total_oil_ton                  float64
total_gas_m3                   float64
total_water_m3                 float64
total_energy_kwh               float64
total_downtime_hours           float64
avg_temperature                float64
avg_pressure                   float64
active_wells                     int64
downtime_pct                   float64
dtype: object

In [7]:
mart_well_kpi.dtypes

well_id                        int64
total_oil_ton                float64
avg_oil_ton                  float64
total_downtime_hours         float64
avg_prod_temperature         float64
avg_prod_pressure            float64
days_count                     int64
downtime_pct                 float64
avg_telemetry_temperature    float64
avg_telemetry_pressure       float64
avg_oil_flow_rate            float64
avg_vibration                float64
well_rank_by_avg_oil         float64
dtype: object

In [8]:
def null_report(df, name):
    report = df.isnull().sum().reset_index()
    report.columns = ["column_name", "null_count"]
    report["table_name"] = name
    report["null_pct"] = (report["null_count"] / len(df) * 100).round(2)
    return report.sort_values("null_count", ascending=False)

null_prod = null_report(production_clean, "production_clean")
null_tel = null_report(well_telemetry_clean, "well_telemetry_clean")
null_daily = null_report(mart_production_daily, "mart_production_daily")
null_kpi = null_report(mart_well_kpi, "mart_well_kpi")

pd.concat([null_prod, null_tel, null_daily, null_kpi], ignore_index=True)

,column_name,null_count,table_name,null_pct
0,temperature,30,production_clean,20.0
1,pressure,30,production_clean,20.0
2,production_id,0,production_clean,0.0
3,well_id,0,production_clean,0.0
4,date,0,production_clean,0.0
5,oil_ton,0,production_clean,0.0
6,gas_m3,0,production_clean,0.0
7,water_m3,0,production_clean,0.0
8,energy_kwh,0,production_clean,0.0
9,downtime_hours,0,production_clean,0.0


In [9]:
duplicates_report = pd.DataFrame({
    "table_name": [
        "production_clean",
        "well_telemetry_clean",
        "mart_production_daily",
        "mart_well_kpi"
    ],
    "duplicate_rows": [
        production_clean.duplicated().sum(),
        well_telemetry_clean.duplicated().sum(),
        mart_production_daily.duplicated().sum(),
        mart_well_kpi.duplicated().sum()
    ]
})

duplicates_report

,table_name,duplicate_rows
0,production_clean,0
1,well_telemetry_clean,0
2,mart_production_daily,0
3,mart_well_kpi,0


In [10]:
quality_checks = {
    "production_negative_oil": int((production_clean["oil_ton"] < 0).sum()),
    "production_downtime_gt_24": int((production_clean["downtime_hours"] > 24).sum()),
    "production_null_temp": int(production_clean["temperature"].isnull().sum()),
    "production_null_pressure": int(production_clean["pressure"].isnull().sum()),
    "telemetry_negative_flow": int((well_telemetry_clean["oil_flow_rate"] < 0).sum()),
    "telemetry_null_temp": int(well_telemetry_clean["temperature"].isnull().sum()),
    "telemetry_null_pressure_out": int(well_telemetry_clean["pressure_out"].isnull().sum()),
    "daily_negative_total_oil": int((mart_production_daily["total_oil_ton"] < 0).sum()),
    "kpi_negative_avg_oil": int((mart_well_kpi["avg_oil_ton"] < 0).sum()),
    "kpi_downtime_pct_gt_100": int((mart_well_kpi["downtime_pct"] > 100).sum()),
}

pd.DataFrame(list(quality_checks.items()), columns=["check_name", "value"])

,check_name,value
0,production_negative_oil,0
1,production_downtime_gt_24,0
2,production_null_temp,30
3,production_null_pressure,30
4,telemetry_negative_flow,0
5,telemetry_null_temp,0
6,telemetry_null_pressure_out,0
7,daily_negative_total_oil,0
8,kpi_negative_avg_oil,0
9,kpi_downtime_pct_gt_100,0


In [11]:
overview = {
    "production_min_date": production_clean["date"].min(),
    "production_max_date": production_clean["date"].max(),
    "production_well_count": production_clean["well_id"].nunique(),
    "telemetry_min_timestamp": well_telemetry_clean["timestamp"].min(),
    "telemetry_max_timestamp": well_telemetry_clean["timestamp"].max(),
    "telemetry_well_count": well_telemetry_clean["well_id"].nunique(),
    "daily_rows": len(mart_production_daily),
    "kpi_rows": len(mart_well_kpi),
}

pd.DataFrame(list(overview.items()), columns=["metric", "value"])

,metric,value
0,production_min_date,2025-10-01 00:00:00
1,production_max_date,2025-10-30 00:00:00
2,production_well_count,5
3,telemetry_min_timestamp,2025-10-01 00:00:00
4,telemetry_max_timestamp,2025-10-01 23:00:00
5,telemetry_well_count,2
6,daily_rows,30
7,kpi_rows,5


In [12]:
production_clean.head(10)

,production_id,well_id,date,oil_ton,gas_m3,water_m3,energy_kwh,downtime_hours,temperature,pressure
0,1,1,2025-10-01,212.4,55200.0,182.3,7450.0,0.5,88.1,120.4
1,31,2,2025-10-01,186.1,49800.0,162.0,6800.0,0.8,84.5,115.4
2,61,3,2025-10-01,121.8,40120.0,121.5,5280.0,2.0,79.4,107.1
3,91,4,2025-10-01,0.0,0.0,0.0,0.0,24.0,NaN,NaN
4,121,5,2025-10-01,197.2,52050.0,165.3,7200.0,0.4,86.5,118.8
5,2,1,2025-10-02,213.8,55320.0,181.9,7490.0,0.3,87.8,121.0
6,32,2,2025-10-02,184.8,49670.0,161.7,6765.0,1.0,84.3,114.8
7,62,3,2025-10-02,120.7,40010.0,120.8,5250.0,2.2,79.7,106.8
8,92,4,2025-10-02,0.0,0.0,0.0,0.0,24.0,NaN,NaN
9,122,5,2025-10-02,198.0,52200.0,166.1,7230.0,0.5,86.7,119.3


In [13]:
well_telemetry_clean.head(10)

,record_id,well_id,timestamp,pump_speed_rpm,pump_current,pressure_in,pressure_out,temperature,vibration,oil_flow_rate,date
0,1,1,2025-10-01 00:00:00,1470.0,58.2,95.3,122.4,88.1,1.4,8.8,2025-10-01
1,2,1,2025-10-01 01:00:00,1468.0,58.5,95.1,122.1,88.2,1.5,8.9,2025-10-01
2,3,1,2025-10-01 02:00:00,1472.0,58.0,94.9,121.8,88.3,1.6,8.7,2025-10-01
3,4,1,2025-10-01 03:00:00,1475.0,58.3,95.0,122.2,88.4,1.4,8.8,2025-10-01
4,5,1,2025-10-01 04:00:00,1473.0,58.1,94.8,121.9,88.2,1.5,8.7,2025-10-01
5,6,1,2025-10-01 05:00:00,1470.0,58.4,95.0,122.3,88.1,1.6,8.9,2025-10-01
6,7,1,2025-10-01 06:00:00,1469.0,58.2,95.1,122.0,88.0,1.5,8.8,2025-10-01
7,8,1,2025-10-01 07:00:00,1474.0,58.6,94.9,121.7,88.3,1.4,8.7,2025-10-01
8,9,1,2025-10-01 08:00:00,1475.0,58.7,95.2,122.5,88.5,1.3,8.9,2025-10-01
9,10,1,2025-10-01 09:00:00,1473.0,58.3,95.0,122.2,88.2,1.4,8.8,2025-10-01


In [14]:
mart_production_daily.head(10)

,date,total_oil_ton,total_gas_m3,total_water_m3,total_energy_kwh,total_downtime_hours,avg_temperature,avg_pressure,active_wells,downtime_pct
0,2025-10-01,717.5,197170.0,631.1,26730.0,27.7,84.625,115.425,5,23.083333
1,2025-10-02,717.3,197200.0,630.5,26735.0,28.0,84.625,115.475,5,23.333333
2,2025-10-03,719.2,197350.0,630.8,26770.0,27.8,84.875,115.500,5,23.166667
3,2025-10-04,722.9,197810.0,627.4,26875.0,26.7,84.375,116.000,5,22.250000
4,2025-10-05,721.0,197620.0,630.1,26810.0,27.5,84.675,115.725,5,22.916667
5,2025-10-06,718.6,197350.0,632.6,26770.0,28.1,84.900,115.475,5,23.416667
6,2025-10-07,714.1,196800.0,635.5,26660.0,29.4,85.275,115.075,5,24.500000
7,2025-10-08,716.4,197090.0,631.9,26725.0,27.6,84.800,115.375,5,23.000000
8,2025-10-09,721.7,197700.0,628.6,26840.0,27.0,84.600,116.000,5,22.500000
9,2025-10-10,718.5,197330.0,631.3,26760.0,27.9,84.850,115.500,5,23.250000


In [15]:
mart_well_kpi.head(10)

,well_id,total_oil_ton,avg_oil_ton,total_downtime_hours,avg_prod_temperature,avg_prod_pressure,days_count,downtime_pct,avg_telemetry_temperature,avg_telemetry_pressure,avg_oil_flow_rate,avg_vibration,well_rank_by_avg_oil
0,1,6394.5,213.150000,14.5,88.206667,120.543333,30,2.013889,88.2875,122.233333,8.808333,1.462500,1.0
1,5,5952.9,198.430000,13.1,86.800000,119.426667,30,1.819444,NaN,NaN,NaN,NaN,2.0
2,2,5574.5,185.816667,21.8,84.486667,115.306667,30,3.027778,84.3500,115.516667,7.537500,1.441667,3.0
3,3,3650.9,121.696667,58.4,79.433333,107.160000,30,8.111111,NaN,NaN,NaN,NaN,4.0
4,4,0.0,0.000000,720.0,NaN,NaN,30,100.000000,NaN,NaN,NaN,NaN,5.0


In [16]:
print("Data quality check completed.")
print(f"production_clean rows: {len(production_clean)}")
print(f"well_telemetry_clean rows: {len(well_telemetry_clean)}")
print(f"mart_production_daily rows: {len(mart_production_daily)}")
print(f"mart_well_kpi rows: {len(mart_well_kpi)}")
print("Critical nulls and invalid negative values were checked.")
print("Curated layer is ready for analytics and ML.")

Data quality check completed.
production_clean rows: 150
well_telemetry_clean rows: 48
mart_production_daily rows: 30
mart_well_kpi rows: 5
Critical nulls and invalid negative values were checked.
Curated layer is ready for analytics and ML.
